In [1]:
import xgboost as xgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import src.train_utils as T
import itertools
import random

In [2]:
df = pd.read_csv('../datasets/dataset_base.csv')
df['date'] = pd.to_datetime(df['date'])

train_df = df[df['date'].dt.year < 2022]

In [3]:
control_df = train_df[['row', 'col', 'date', 'delta_pm25_t', 'delta_pm25_t+1']]
control_df.head()

,row,col,date,delta_pm25_t,delta_pm25_t+1
0,0,0,2018-07-05,0.302968,0.601407
1,0,0,2018-07-06,0.601407,0.038002
2,0,0,2018-07-07,0.038002,0.556278
3,0,0,2018-07-08,0.556278,-0.491020
4,0,0,2018-07-09,-0.491020,1.041161


### Randomized Hyperparameter Search

In [ ]:
param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'n_estimators': [300, 500]
}

# 2. Generate all hyperparameter combinations
param_list = list(itertools.product(
    param_grid['max_depth'],
    param_grid['learning_rate'],
    param_grid['subsample'],
    param_grid['colsample_bytree'],
    param_grid['n_estimators']
))

# 3. Randomly sample a subset
n_samples = 20  # or however many trials you want
random.seed(191)  # reproducibility
sampled_params = random.sample(param_list, min(n_samples, len(param_list)))

# 3. Initialize tracking
results = []

# 4. For each hyperparameter setting
for params in sampled_params:
    max_depth, learning_rate, subsample, colsample_bytree, n_estimators = params

    model = xgb.XGBRegressor(
        objective='reg:squarederror',
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        n_estimators=n_estimators,
        random_state=191
    )

    rmse_scores, mae_scores = T.rolling_window_cv(
        train_days=365 * 2,
        gap_days=21,
        test_days=60,
        df=control_df,
        model=model
    )

    mean_rmse = np.mean(rmse_scores)
    se_rmse = np.std(rmse_scores)

    mean_mae = np.mean(mae_scores)
    se_mae = np.std(mae_scores)

    print("Summary of Performance Statistics")
    print(f"RMSE: {mean_rmse} +/- {se_rmse}")
    print(f"MAE: {mean_mae} +/- {se_mae}")

    results.append((params, mean_rmse))

best_params, best_score = min(results, key=lambda x: x[1])
print("Best parameters:", best_params)
print("Best average RMSE:", best_score)

Fold 1
2018-07-05 00:00:00 2020-07-19 00:00:00
2020-08-10 00:00:00 2020-10-08 00:00:00
RMSE: 1.6610864331215185
MAE: 1.0046119332320411
Fold 2
2018-09-03 00:00:00 2020-09-17 00:00:00
2020-10-09 00:00:00 2020-12-07 00:00:00
RMSE: 5.265336976457552
MAE: 2.177977929955929
Fold 3
2018-11-02 00:00:00 2020-11-16 00:00:00
2020-12-08 00:00:00 2021-02-05 00:00:00
RMSE: 7.309499283029077
MAE: 5.279105736917975
Fold 4
2019-01-01 00:00:00 2021-01-15 00:00:00
2021-02-06 00:00:00 2021-04-18 00:00:00
RMSE: 13.477285619805523
MAE: 9.305458266236561
Fold 5
2019-03-04 00:00:00 2021-03-28 00:00:00
2021-04-19 00:00:00 2021-06-17 00:00:00
RMSE: 4.849226066736387
MAE: 2.6855442538824
Fold 6
2019-05-03 00:00:00 2021-05-27 00:00:00
2021-06-18 00:00:00 2021-08-16 00:00:00
RMSE: 1.052554341929309
MAE: 0.8185394821145112
Fold 7
2019-07-02 00:00:00 2021-07-26 00:00:00
2021-08-17 00:00:00 2021-10-15 00:00:00
RMSE: 1.2843955604813735
MAE: 0.9224955986220128
Fold 8
2019-08-31 00:00:00 2021-09-24 00:00:00
2021-10-16 

Best Params: (4, 0.1, 0.8, 0.8, 500)

In [6]:
params = {'max_depth': 4, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'n_estimators': 500}

In [47]:
exp_1_df

,row,col,date,pm25_t,u_wind_10m,v_wind_10m,dew_temp_2m,temp_2m,surf_pressure,precip_sum,...,delta_pm25_t_r1_c-2,delta_pm25_t_r1_c-1,delta_pm25_t_r1_c0,delta_pm25_t_r1_c1,delta_pm25_t_r1_c2,delta_pm25_t_r2_c-2,delta_pm25_t_r2_c-1,delta_pm25_t_r2_c0,delta_pm25_t_r2_c1,delta_pm25_t_r2_c2
112320,2,2,2018-07-05,8.506299,1.099417,0.845259,292.53455,297.36017,88500.480,1.085705e-03,...,0.506664,0.420908,1.175132,1.152598,0.412996,0.275500,-0.419762,0.973524,0.522349,0.302968
112321,2,2,2018-07-06,7.291668,0.536112,0.847002,292.94757,296.90536,88492.290,2.183610e-03,...,-0.233806,-0.559418,-1.652848,-1.761432,0.167479,-0.186723,-0.095932,-1.125940,-1.001619,0.601407
112322,2,2,2018-07-07,8.418976,0.653129,0.892963,293.46637,296.09024,88523.930,1.098063e-02,...,-0.039803,0.708406,1.128092,1.741480,0.411181,0.311488,0.669046,0.587489,1.064866,0.038002
112323,2,2,2018-07-08,9.029675,0.758944,0.372737,293.11792,295.39346,88557.590,7.403880e-03,...,0.436303,0.311186,0.570811,0.926645,1.613415,0.208978,0.167542,0.058055,0.060918,0.556278
112324,2,2,2018-07-09,9.116474,0.735163,0.694967,292.74075,295.87690,88641.150,1.645568e-03,...,0.123422,-0.060415,-0.185492,-0.514772,-1.492460,0.251034,0.176971,0.625493,0.339427,-0.491020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2303803,41,41,2021-12-27,21.843426,-1.568173,-0.212587,287.75488,293.73010,95798.210,2.097818e-05,...,-9.537959,-12.137171,-8.159606,-6.374832,-6.324257,-9.670241,-14.552199,-10.942500,-5.915656,-5.896442
2303804,41,41,2021-12-28,18.682210,-1.002234,-0.391176,286.59882,293.61517,95750.990,4.410744e-07,...,-1.771536,-2.445273,-3.504840,-4.787384,-5.964996,-0.231837,0.978988,-0.261930,-5.346428,-5.612656
2303805,41,41,2021-12-29,21.692741,-1.128024,-0.570042,287.97546,294.03458,95779.375,4.266560e-06,...,2.188764,3.974985,3.253660,3.584790,2.722079,1.806360,2.974662,3.822099,4.014383,3.273466
2303806,41,41,2021-12-30,19.682276,-1.364955,-0.540719,287.61550,293.62190,95875.930,1.715147e-06,...,-3.071544,-3.512343,-1.171859,-2.556478,-1.015366,-3.052140,-2.895143,-2.726351,-2.949779,-1.906230


In [84]:
exp_1_df = train_df.copy()

base_df = exp_1_df.copy()
shifts = []
for row_shift in [-2, -1, 0, 1, 2]:
    for col_shift in [-2, -1, 0, 1, 2]:
        if not (row_shift == 0 and col_shift == 0):  # skip center cell
            shifts.append((row_shift, col_shift))

for row_shift, col_shift in shifts:
    shifted = base_df.copy()
    shifted['row'] += row_shift
    shifted['col'] += col_shift

    # Naming convention
    col_name_pm = f'pm25_t_r{-row_shift}_c{-col_shift}'
    col_name_delta = f'delta_pm25_t_r{-row_shift}_c{-col_shift}'

    base_df = base_df.merge(
        shifted[['row', 'col', 'date', 'pm25_t', 'delta_pm25_t']].rename(columns={'pm25_t': col_name_pm, 'delta_pm25_t': col_name_delta}),
        on=['row', 'col', 'date'],
        how='left'
    )

exp_1_df = base_df.dropna()


In [87]:
exp_1_df.to_csv('exp_1_dataset.csv', index=False)

## Control 1: Current Pixel Only

In [91]:
control_df = exp_1_df[['row', 'col', 'date', 'delta_pm25_t', 'delta_pm25_t+1']]
control_df.head()

,row,col,date,delta_pm25_t,delta_pm25_t+1
112320,2,2,2018-07-05,0.774012,-1.214631
112321,2,2,2018-07-06,-1.214631,1.127308
112322,2,2,2018-07-07,1.127308,0.610699
112323,2,2,2018-07-08,0.610699,0.086799
112324,2,2,2018-07-09,0.086799,0.337736


In [92]:
model = xgb.XGBRegressor(
        objective='reg:squarederror',
        **params,
        random_state=191
    )

rmse_scores, mae_scores = T.rolling_window_cv(
    train_days=365 * 2,
    gap_days=21,
    test_days=60,
    df=control_df,
    model=model
)

mean_rmse = np.mean(rmse_scores)
se_rmse = np.std(rmse_scores)

mean_mae = np.mean(mae_scores)
se_mae = np.std(mae_scores)

print("Summary of Performance Statistics")
print(f"RMSE: {mean_rmse} +/- {se_rmse}")
print(f"MAE: {mean_mae} +/- {se_mae}")

Fold 1
2018-07-05 00:00:00 2020-07-19 00:00:00
2020-08-10 00:00:00 2020-10-08 00:00:00
RMSE: 1.5897928502961691
MAE: 1.001991590397039
Fold 2
2018-09-03 00:00:00 2020-09-17 00:00:00
2020-10-09 00:00:00 2020-12-07 00:00:00
RMSE: 5.618287266480897
MAE: 2.227218569097601
Fold 3
2018-11-02 00:00:00 2020-11-16 00:00:00
2020-12-08 00:00:00 2021-02-05 00:00:00
RMSE: 7.36944173771286
MAE: 5.327215970615451
Fold 4
2019-01-01 00:00:00 2021-01-15 00:00:00
2021-02-06 00:00:00 2021-04-18 00:00:00
RMSE: 13.494239477873599
MAE: 9.335713399547238
Fold 5
2019-03-04 00:00:00 2021-03-28 00:00:00
2021-04-19 00:00:00 2021-06-17 00:00:00
RMSE: 4.673510526115822
MAE: 2.6647422491553363
Fold 6
2019-05-03 00:00:00 2021-05-27 00:00:00
2021-06-18 00:00:00 2021-08-16 00:00:00
RMSE: 1.0643947649868184
MAE: 0.827335253199704
Fold 7
2019-07-02 00:00:00 2021-07-26 00:00:00
2021-08-17 00:00:00 2021-10-15 00:00:00
RMSE: 1.2824402274431383
MAE: 0.9224717113617616
Fold 8
2019-08-31 00:00:00 2021-09-24 00:00:00
2021-10-16

## + 4 Neighbors

In [93]:
four_neighbors_df = exp_1_df[['row', 'col', 'date', 'delta_pm25_t', 
                              'delta_pm25_t_r1_c0', 'delta_pm25_t_r-1_c0',
                              'delta_pm25_t_r0_c1', 'delta_pm25_t_r0_c-1',
                              'delta_pm25_t+1']]
four_neighbors_df.head()

,row,col,date,delta_pm25_t,delta_pm25_t_r1_c0,delta_pm25_t_r-1_c0,delta_pm25_t_r0_c1,delta_pm25_t_r0_c-1,delta_pm25_t+1
112320,2,2,2018-07-05,0.774012,0.198054,1.175132,0.815492,1.526387,-1.214631
112321,2,2,2018-07-06,-1.214631,-0.564943,-1.652848,-0.376699,-2.001883,1.127308
112322,2,2,2018-07-07,1.127308,1.057073,1.128092,0.389675,1.450109,0.610699
112323,2,2,2018-07-08,0.610699,0.560488,0.570811,0.433100,0.941031,0.086799
112324,2,2,2018-07-09,0.086799,-0.156699,-0.185492,0.020303,-0.670372,0.337736


In [94]:
model = xgb.XGBRegressor(
        objective='reg:squarederror',
        **params,
        random_state=191
    )

rmse_scores, mae_scores = T.rolling_window_cv(
    train_days=365 * 2,
    gap_days=21,
    test_days=60,
    df=four_neighbors_df,
    model=model
)

mean_rmse = np.mean(rmse_scores)
se_rmse = np.std(rmse_scores)

mean_mae = np.mean(mae_scores)
se_mae = np.std(mae_scores)

print("Summary of Performance Statistics")
print(f"RMSE: {mean_rmse} +/- {se_rmse}")
print(f"MAE: {mean_mae} +/- {se_mae}")

Fold 1
2018-07-05 00:00:00 2020-07-19 00:00:00
2020-08-10 00:00:00 2020-10-08 00:00:00
RMSE: 1.5785777921831667
MAE: 1.0013792503747463
Fold 2
2018-09-03 00:00:00 2020-09-17 00:00:00
2020-10-09 00:00:00 2020-12-07 00:00:00
RMSE: 5.613008644961647
MAE: 2.229501816930823
Fold 3
2018-11-02 00:00:00 2020-11-16 00:00:00
2020-12-08 00:00:00 2021-02-05 00:00:00
RMSE: 7.3442446642488015
MAE: 5.312837267287146
Fold 4
2019-01-01 00:00:00 2021-01-15 00:00:00
2021-02-06 00:00:00 2021-04-18 00:00:00
RMSE: 13.529997485038093
MAE: 9.344804929425639
Fold 5
2019-03-04 00:00:00 2021-03-28 00:00:00
2021-04-19 00:00:00 2021-06-17 00:00:00
RMSE: 4.645750868302276
MAE: 2.6533022790397864
Fold 6
2019-05-03 00:00:00 2021-05-27 00:00:00
2021-06-18 00:00:00 2021-08-16 00:00:00
RMSE: 1.0743554391322407
MAE: 0.8379624698442981
Fold 7
2019-07-02 00:00:00 2021-07-26 00:00:00
2021-08-17 00:00:00 2021-10-15 00:00:00
RMSE: 1.2847795423092345
MAE: 0.9273106680343102
Fold 8
2019-08-31 00:00:00 2021-09-24 00:00:00
2021-1

## + 3x3

In [96]:
three_df = exp_1_df[['row', 'col', 'date', 'delta_pm25_t', 
                    'delta_pm25_t_r1_c0', 'delta_pm25_t_r-1_c0',
                    'delta_pm25_t_r0_c1', 'delta_pm25_t_r0_c-1',
                    'delta_pm25_t_r1_c1', 'delta_pm25_t_r1_c-1',
                    'delta_pm25_t_r-1_c1', 'delta_pm25_t_r-1_c-1',
                    'delta_pm25_t+1']]

three_df.head()

,row,col,date,delta_pm25_t,delta_pm25_t_r1_c0,delta_pm25_t_r-1_c0,delta_pm25_t_r0_c1,delta_pm25_t_r0_c-1,delta_pm25_t_r1_c1,delta_pm25_t_r1_c-1,delta_pm25_t_r-1_c1,delta_pm25_t_r-1_c-1,delta_pm25_t+1
112320,2,2,2018-07-05,0.774012,0.198054,1.175132,0.815492,1.526387,0.119964,-0.102599,0.420908,1.152598,-1.214631
112321,2,2,2018-07-06,-1.214631,-0.564943,-1.652848,-0.376699,-2.001883,0.452158,-0.351426,-0.559418,-1.761432,1.127308
112322,2,2,2018-07-07,1.127308,1.057073,1.128092,0.389675,1.450109,0.063593,1.017430,0.708406,1.741480,0.610699
112323,2,2,2018-07-08,0.610699,0.560488,0.570811,0.433100,0.941031,0.888477,0.539345,0.311186,0.926645,0.086799
112324,2,2,2018-07-09,0.086799,-0.156699,-0.185492,0.020303,-0.670372,-0.533840,-0.117224,-0.060415,-0.514772,0.337736


In [97]:
model = xgb.XGBRegressor(
        objective='reg:squarederror',
        **params,
        random_state=191
    )

rmse_scores, mae_scores = T.rolling_window_cv(
    train_days=365 * 2,
    gap_days=21,
    test_days=60,
    df=three_df,
    model=model
)

mean_rmse = np.mean(rmse_scores)
se_rmse = np.std(rmse_scores)

mean_mae = np.mean(mae_scores)
se_mae = np.std(mae_scores)

print("Summary of Performance Statistics")
print(f"RMSE: {mean_rmse} +/- {se_rmse}")
print(f"MAE: {mean_mae} +/- {se_mae}")

Fold 1
2018-07-05 00:00:00 2020-07-19 00:00:00
2020-08-10 00:00:00 2020-10-08 00:00:00
RMSE: 1.5759573827425137
MAE: 0.9997676154662491
Fold 2
2018-09-03 00:00:00 2020-09-17 00:00:00
2020-10-09 00:00:00 2020-12-07 00:00:00
RMSE: 5.579813847220417
MAE: 2.2268854917188925
Fold 3
2018-11-02 00:00:00 2020-11-16 00:00:00
2020-12-08 00:00:00 2021-02-05 00:00:00
RMSE: 7.337851900922219
MAE: 5.3086458687488
Fold 4
2019-01-01 00:00:00 2021-01-15 00:00:00
2021-02-06 00:00:00 2021-04-18 00:00:00
RMSE: 13.536861314474677
MAE: 9.34335377605413
Fold 5
2019-03-04 00:00:00 2021-03-28 00:00:00
2021-04-19 00:00:00 2021-06-17 00:00:00
RMSE: 4.629073992102295
MAE: 2.6437329033334773
Fold 6
2019-05-03 00:00:00 2021-05-27 00:00:00
2021-06-18 00:00:00 2021-08-16 00:00:00
RMSE: 1.0716861639465798
MAE: 0.8357961484610211
Fold 7
2019-07-02 00:00:00 2021-07-26 00:00:00
2021-08-17 00:00:00 2021-10-15 00:00:00
RMSE: 1.283027104595729
MAE: 0.9259225228307828
Fold 8
2019-08-31 00:00:00 2021-09-24 00:00:00
2021-10-16

## + 12 Neighbors

In [98]:
twelve_neighbors_df = exp_1_df[['row', 'col', 'date', 'delta_pm25_t', 
                                'delta_pm25_t_r1_c0', 'delta_pm25_t_r-1_c0',
                                'delta_pm25_t_r0_c1', 'delta_pm25_t_r0_c-1',
                                'delta_pm25_t_r1_c1', 'delta_pm25_t_r1_c-1',
                                'delta_pm25_t_r-1_c1', 'delta_pm25_t_r-1_c-1',
                                'delta_pm25_t_r-2_c0', 'delta_pm25_t_r2_c0',
                                'delta_pm25_t_r0_c-2', 'delta_pm25_t_r0_c2',
                                'delta_pm25_t+1']]
twelve_neighbors_df.head()

,row,col,date,delta_pm25_t,delta_pm25_t_r1_c0,delta_pm25_t_r-1_c0,delta_pm25_t_r0_c1,delta_pm25_t_r0_c-1,delta_pm25_t_r1_c1,delta_pm25_t_r1_c-1,delta_pm25_t_r-1_c1,delta_pm25_t_r-1_c-1,delta_pm25_t_r-2_c0,delta_pm25_t_r2_c0,delta_pm25_t_r0_c-2,delta_pm25_t_r0_c2,delta_pm25_t+1
112320,2,2,2018-07-05,0.774012,0.198054,1.175132,0.815492,1.526387,0.119964,-0.102599,0.420908,1.152598,0.973524,0.745076,0.531390,-0.076599,-1.214631
112321,2,2,2018-07-06,-1.214631,-0.564943,-1.652848,-0.376699,-2.001883,0.452158,-0.351426,-0.559418,-1.761432,-1.125940,-0.788587,-0.279049,0.366528,1.127308
112322,2,2,2018-07-07,1.127308,1.057073,1.128092,0.389675,1.450109,0.063593,1.017430,0.708406,1.741480,0.587489,0.626072,0.490491,-0.134154,0.610699
112323,2,2,2018-07-08,0.610699,0.560488,0.570811,0.433100,0.941031,0.888477,0.539345,0.311186,0.926645,0.058055,1.012921,1.268657,0.875981,0.086799
112324,2,2,2018-07-09,0.086799,-0.156699,-0.185492,0.020303,-0.670372,-0.533840,-0.117224,-0.060415,-0.514772,0.625493,-0.704665,-1.068175,-0.195938,0.337736


In [99]:
model = xgb.XGBRegressor(
        objective='reg:squarederror',
        **params,
        random_state=191
    )

rmse_scores, mae_scores = T.rolling_window_cv(
    train_days=365 * 2,
    gap_days=21,
    test_days=60,
    df=twelve_neighbors_df,
    model=model
)

mean_rmse = np.mean(rmse_scores)
se_rmse = np.std(rmse_scores)

mean_mae = np.mean(mae_scores)
se_mae = np.std(mae_scores)

print("Summary of Performance Statistics")
print(f"RMSE: {mean_rmse} +/- {se_rmse}")
print(f"MAE: {mean_mae} +/- {se_mae}")

Fold 1
2018-07-05 00:00:00 2020-07-19 00:00:00
2020-08-10 00:00:00 2020-10-08 00:00:00
RMSE: 1.5676567525416538
MAE: 0.9957675864819716
Fold 2
2018-09-03 00:00:00 2020-09-17 00:00:00
2020-10-09 00:00:00 2020-12-07 00:00:00
RMSE: 5.55176015529906
MAE: 2.2277422519410846
Fold 3
2018-11-02 00:00:00 2020-11-16 00:00:00
2020-12-08 00:00:00 2021-02-05 00:00:00
RMSE: 7.328219481336435
MAE: 5.30266722826136
Fold 4
2019-01-01 00:00:00 2021-01-15 00:00:00
2021-02-06 00:00:00 2021-04-18 00:00:00
RMSE: 13.510267663276371
MAE: 9.332887255981692
Fold 5
2019-03-04 00:00:00 2021-03-28 00:00:00
2021-04-19 00:00:00 2021-06-17 00:00:00
RMSE: 4.607239146482127
MAE: 2.6302593583345466
Fold 6
2019-05-03 00:00:00 2021-05-27 00:00:00
2021-06-18 00:00:00 2021-08-16 00:00:00
RMSE: 1.0649742908195405
MAE: 0.8310171363549559
Fold 7
2019-07-02 00:00:00 2021-07-26 00:00:00
2021-08-17 00:00:00 2021-10-15 00:00:00
RMSE: 1.2787394698082901
MAE: 0.9238036120911153
Fold 8
2019-08-31 00:00:00 2021-09-24 00:00:00
2021-10-

## + 5x5

In [101]:
five_df = exp_1_df[['row', 'col', 'date', 'delta_pm25_t',
                    'delta_pm25_t_r-2_c-2', 'delta_pm25_t_r-2_c-1', 'delta_pm25_t_r-2_c0', 'delta_pm25_t_r-2_c1', 'delta_pm25_t_r-2_c2',
                    'delta_pm25_t_r-1_c-2', 'delta_pm25_t_r-1_c-1', 'delta_pm25_t_r-1_c0', 'delta_pm25_t_r-1_c1', 'delta_pm25_t_r-1_c2',
                    'delta_pm25_t_r0_c-2', 'delta_pm25_t_r0_c-1', 'delta_pm25_t_r0_c1', 'delta_pm25_t_r0_c2',
                    'delta_pm25_t_r1_c-2', 'delta_pm25_t_r1_c-1', 'delta_pm25_t_r1_c0', 'delta_pm25_t_r1_c1', 'delta_pm25_t_r1_c2',
                    'delta_pm25_t_r2_c-2', 'delta_pm25_t_r2_c-1', 'delta_pm25_t_r2_c0', 'delta_pm25_t_r2_c1', 'delta_pm25_t_r2_c2',
                    'delta_pm25_t+1'
]]

five_df.head()

,row,col,date,delta_pm25_t,delta_pm25_t_r-2_c-2,delta_pm25_t_r-2_c-1,delta_pm25_t_r-2_c0,delta_pm25_t_r-2_c1,delta_pm25_t_r-2_c2,delta_pm25_t_r-1_c-2,...,delta_pm25_t_r1_c-1,delta_pm25_t_r1_c0,delta_pm25_t_r1_c1,delta_pm25_t_r1_c2,delta_pm25_t_r2_c-2,delta_pm25_t_r2_c-1,delta_pm25_t_r2_c0,delta_pm25_t_r2_c1,delta_pm25_t_r2_c2,delta_pm25_t+1
112320,2,2,2018-07-05,0.774012,0.302968,0.522349,0.973524,-0.419762,0.275500,0.412996,...,-0.102599,0.198054,0.119964,0.473723,0.354453,0.733201,0.745076,0.145992,0.144983,-1.214631
112321,2,2,2018-07-06,-1.214631,0.601407,-1.001619,-1.125940,-0.095932,-0.186723,0.167479,...,-0.351426,-0.564943,0.452158,0.313466,-0.011651,-0.960510,-0.788587,0.048550,0.975503,1.127308
112322,2,2,2018-07-07,1.127308,0.038002,1.064866,0.587489,0.669046,0.311488,0.411181,...,1.017430,1.057073,0.063593,-0.459555,0.387937,1.195193,0.626072,0.028603,-0.755823,0.610699
112323,2,2,2018-07-08,0.610699,0.556278,0.060918,0.058055,0.167542,0.208978,1.613415,...,0.539345,0.560488,0.888477,1.368596,0.601076,0.385827,1.012921,0.865992,1.554551,0.086799
112324,2,2,2018-07-09,0.086799,-0.491020,0.339427,0.625493,0.176971,0.251034,-1.492460,...,-0.117224,-0.156699,-0.533840,-0.700388,-0.497408,-0.014912,-0.704665,-0.464011,-1.017161,0.337736


In [102]:
model = xgb.XGBRegressor(
        objective='reg:squarederror',
        **params,
        random_state=191
    )

rmse_scores, mae_scores = T.rolling_window_cv(
    train_days=365 * 2,
    gap_days=21,
    test_days=60,
    df=five_df,
    model=model
)

mean_rmse = np.mean(rmse_scores)
se_rmse = np.std(rmse_scores)

mean_mae = np.mean(mae_scores)
se_mae = np.std(mae_scores)

print("Summary of Performance Statistics")
print(f"RMSE: {mean_rmse} +/- {se_rmse}")
print(f"MAE: {mean_mae} +/- {se_mae}")

Fold 1
2018-07-05 00:00:00 2020-07-19 00:00:00
2020-08-10 00:00:00 2020-10-08 00:00:00
RMSE: 1.5598022066688282
MAE: 0.9893444888842248
Fold 2
2018-09-03 00:00:00 2020-09-17 00:00:00
2020-10-09 00:00:00 2020-12-07 00:00:00
RMSE: 5.5091415909438455
MAE: 2.2271945979154117
Fold 3
2018-11-02 00:00:00 2020-11-16 00:00:00
2020-12-08 00:00:00 2021-02-05 00:00:00
RMSE: 7.319231141730841
MAE: 5.293315727230608
Fold 4
2019-01-01 00:00:00 2021-01-15 00:00:00
2021-02-06 00:00:00 2021-04-18 00:00:00
RMSE: 13.49614243020087
MAE: 9.32217443028623
Fold 5
2019-03-04 00:00:00 2021-03-28 00:00:00
2021-04-19 00:00:00 2021-06-17 00:00:00
RMSE: 4.572752406523888
MAE: 2.6124138074858134
Fold 6
2019-05-03 00:00:00 2021-05-27 00:00:00
2021-06-18 00:00:00 2021-08-16 00:00:00
RMSE: 1.0590410167322535
MAE: 0.8264982256104146
Fold 7
2019-07-02 00:00:00 2021-07-26 00:00:00
2021-08-17 00:00:00 2021-10-15 00:00:00
RMSE: 1.272808640463271
MAE: 0.9195810767088981
Fold 8
2019-08-31 00:00:00 2021-09-24 00:00:00
2021-10-

## Control 2: Exo + Current Pixel Only

In [110]:
control_2_df = train_df[['row', 'col', 'date', 'pm25_t', 'u_wind_10m', 'v_wind_10m',
                        'dew_temp_2m', 'temp_2m', 'surf_pressure', 'precip_sum', 'frp',
                        'elevation', 'delta_pm25_t+1']]

control_2_df.head()

,row,col,date,pm25_t,u_wind_10m,v_wind_10m,dew_temp_2m,temp_2m,surf_pressure,precip_sum,frp,elevation,delta_pm25_t+1
0,0,0,2018-07-05,8.317946,1.432791,0.080936,291.41534,295.22916,85304.766,0.003423,0.0,1360.0,0.601407
1,0,0,2018-07-06,8.919353,0.748148,0.110430,291.74884,295.01360,85287.414,0.007418,0.0,1360.0,0.038002
2,0,0,2018-07-07,8.957355,0.922457,0.196369,292.11148,294.27896,85314.555,0.013237,0.0,1360.0,0.556278
3,0,0,2018-07-08,9.513633,1.040234,0.047216,291.94238,293.10020,85344.050,0.011159,0.0,1360.0,-0.491020
4,0,0,2018-07-09,9.022613,1.154719,0.083273,290.86307,293.92580,85426.945,0.001921,0.0,1360.0,1.041161


In [111]:
model = xgb.XGBRegressor(
        objective='reg:squarederror',
        **params,
        random_state=191
    )

rmse_scores, mae_scores = T.rolling_window_cv(
    train_days=365 * 2,
    gap_days=21,
    test_days=60,
    df=control_2_df,
    model=model
)

mean_rmse = np.mean(rmse_scores)
se_rmse = np.std(rmse_scores)

mean_mae = np.mean(mae_scores)
se_mae = np.std(mae_scores)

print("Summary of Performance Statistics")
print(f"RMSE: {mean_rmse} +/- {se_rmse}")
print(f"MAE: {mean_mae} +/- {se_mae}")

Fold 1
2018-07-05 00:00:00 2020-07-19 00:00:00
2020-08-10 00:00:00 2020-10-08 00:00:00
RMSE: 1.6247664205195806
MAE: 0.9944899016250245
Fold 2
2018-09-03 00:00:00 2020-09-17 00:00:00
2020-10-09 00:00:00 2020-12-07 00:00:00
RMSE: 4.900930010792188
MAE: 2.294106657863162
Fold 3
2018-11-02 00:00:00 2020-11-16 00:00:00
2020-12-08 00:00:00 2021-02-05 00:00:00
RMSE: 6.988699657193306
MAE: 5.097970069094706
Fold 4
2019-01-01 00:00:00 2021-01-15 00:00:00
2021-02-06 00:00:00 2021-04-18 00:00:00
RMSE: 12.957978759707443
MAE: 9.163356840585989
Fold 5
2019-03-04 00:00:00 2021-03-28 00:00:00
2021-04-19 00:00:00 2021-06-17 00:00:00
RMSE: 4.129106259624338
MAE: 2.340181888558889
Fold 6
2019-05-03 00:00:00 2021-05-27 00:00:00
2021-06-18 00:00:00 2021-08-16 00:00:00
RMSE: 1.0145065325252662
MAE: 0.7834504441368625
Fold 7
2019-07-02 00:00:00 2021-07-26 00:00:00
2021-08-17 00:00:00 2021-10-15 00:00:00
RMSE: 1.3507151692618595
MAE: 0.9892809559238133
Fold 8
2019-08-31 00:00:00 2021-09-24 00:00:00
2021-10-

## Exo + 4 Dir

In [112]:
four_neighbors_exo_df = exp_1_df[['row', 'col', 'date', 'pm25_t', 'u_wind_10m', 'v_wind_10m',
                                    'dew_temp_2m', 'temp_2m', 'surf_pressure', 'precip_sum', 'frp',
                                    'elevation', 'delta_pm25_t', 
                                    'delta_pm25_t_r1_c0', 'delta_pm25_t_r-1_c0',
                                    'delta_pm25_t_r0_c1', 'delta_pm25_t_r0_c-1',
                                    'delta_pm25_t+1']]
four_neighbors_exo_df.head()

,row,col,date,pm25_t,u_wind_10m,v_wind_10m,dew_temp_2m,temp_2m,surf_pressure,precip_sum,frp,elevation,delta_pm25_t,delta_pm25_t_r1_c0,delta_pm25_t_r-1_c0,delta_pm25_t_r0_c1,delta_pm25_t_r0_c-1,delta_pm25_t+1
112320,2,2,2018-07-05,8.506299,1.099417,0.845259,292.53455,297.36017,88500.48,0.001086,0.0,1018.0,0.774012,0.198054,1.175132,0.815492,1.526387,-1.214631
112321,2,2,2018-07-06,7.291668,0.536112,0.847002,292.94757,296.90536,88492.29,0.002184,0.0,1018.0,-1.214631,-0.564943,-1.652848,-0.376699,-2.001883,1.127308
112322,2,2,2018-07-07,8.418976,0.653129,0.892963,293.46637,296.09024,88523.93,0.010981,0.0,1018.0,1.127308,1.057073,1.128092,0.389675,1.450109,0.610699
112323,2,2,2018-07-08,9.029675,0.758944,0.372737,293.11792,295.39346,88557.59,0.007404,0.0,1018.0,0.610699,0.560488,0.570811,0.433100,0.941031,0.086799
112324,2,2,2018-07-09,9.116474,0.735163,0.694967,292.74075,295.87690,88641.15,0.001646,0.0,1018.0,0.086799,-0.156699,-0.185492,0.020303,-0.670372,0.337736


In [113]:
model = xgb.XGBRegressor(
        objective='reg:squarederror',
        **params,
        random_state=191
    )

rmse_scores, mae_scores = T.rolling_window_cv(
    train_days=365 * 2,
    gap_days=21,
    test_days=60,
    df=four_neighbors_exo_df,
    model=model
)

mean_rmse = np.mean(rmse_scores)
se_rmse = np.std(rmse_scores)

mean_mae = np.mean(mae_scores)
se_mae = np.std(mae_scores)

print("Summary of Performance Statistics")
print(f"RMSE: {mean_rmse} +/- {se_rmse}")
print(f"MAE: {mean_mae} +/- {se_mae}")

Fold 1
2018-07-05 00:00:00 2020-07-19 00:00:00
2020-08-10 00:00:00 2020-10-08 00:00:00
RMSE: 1.5564052256111849
MAE: 0.994689609480535
Fold 2
2018-09-03 00:00:00 2020-09-17 00:00:00
2020-10-09 00:00:00 2020-12-07 00:00:00
RMSE: 5.08042626188442
MAE: 2.3219789122029324
Fold 3
2018-11-02 00:00:00 2020-11-16 00:00:00
2020-12-08 00:00:00 2021-02-05 00:00:00
RMSE: 7.057507430296393
MAE: 5.183134363089853
Fold 4
2019-01-01 00:00:00 2021-01-15 00:00:00
2021-02-06 00:00:00 2021-04-18 00:00:00
RMSE: 12.824998099169092
MAE: 9.165941950010705
Fold 5
2019-03-04 00:00:00 2021-03-28 00:00:00
2021-04-19 00:00:00 2021-06-17 00:00:00
RMSE: 4.068097412530483
MAE: 2.3618302486838076
Fold 6
2019-05-03 00:00:00 2021-05-27 00:00:00
2021-06-18 00:00:00 2021-08-16 00:00:00
RMSE: 1.0000673962786788
MAE: 0.7733748642687686
Fold 7
2019-07-02 00:00:00 2021-07-26 00:00:00
2021-08-17 00:00:00 2021-10-15 00:00:00
RMSE: 1.3120897113190704
MAE: 0.9622655549339457
Fold 8
2019-08-31 00:00:00 2021-09-24 00:00:00
2021-10-

## 3x3 + Exo

In [114]:
three_exo_df = exp_1_df[['row', 'col', 'date', 'pm25_t', 'u_wind_10m', 'v_wind_10m',
                        'dew_temp_2m', 'temp_2m', 'surf_pressure', 'precip_sum', 'frp',
                        'elevation', 'delta_pm25_t',  
                        'delta_pm25_t_r1_c0', 'delta_pm25_t_r-1_c0',
                        'delta_pm25_t_r0_c1', 'delta_pm25_t_r0_c-1',
                        'delta_pm25_t_r1_c1', 'delta_pm25_t_r1_c-1',
                        'delta_pm25_t_r-1_c1', 'delta_pm25_t_r-1_c-1',
                        'delta_pm25_t+1']]

three_exo_df.head()

,row,col,date,pm25_t,u_wind_10m,v_wind_10m,dew_temp_2m,temp_2m,surf_pressure,precip_sum,...,delta_pm25_t,delta_pm25_t_r1_c0,delta_pm25_t_r-1_c0,delta_pm25_t_r0_c1,delta_pm25_t_r0_c-1,delta_pm25_t_r1_c1,delta_pm25_t_r1_c-1,delta_pm25_t_r-1_c1,delta_pm25_t_r-1_c-1,delta_pm25_t+1
112320,2,2,2018-07-05,8.506299,1.099417,0.845259,292.53455,297.36017,88500.48,0.001086,...,0.774012,0.198054,1.175132,0.815492,1.526387,0.119964,-0.102599,0.420908,1.152598,-1.214631
112321,2,2,2018-07-06,7.291668,0.536112,0.847002,292.94757,296.90536,88492.29,0.002184,...,-1.214631,-0.564943,-1.652848,-0.376699,-2.001883,0.452158,-0.351426,-0.559418,-1.761432,1.127308
112322,2,2,2018-07-07,8.418976,0.653129,0.892963,293.46637,296.09024,88523.93,0.010981,...,1.127308,1.057073,1.128092,0.389675,1.450109,0.063593,1.017430,0.708406,1.741480,0.610699
112323,2,2,2018-07-08,9.029675,0.758944,0.372737,293.11792,295.39346,88557.59,0.007404,...,0.610699,0.560488,0.570811,0.433100,0.941031,0.888477,0.539345,0.311186,0.926645,0.086799
112324,2,2,2018-07-09,9.116474,0.735163,0.694967,292.74075,295.87690,88641.15,0.001646,...,0.086799,-0.156699,-0.185492,0.020303,-0.670372,-0.533840,-0.117224,-0.060415,-0.514772,0.337736


In [115]:
model = xgb.XGBRegressor(
        objective='reg:squarederror',
        **params,
        random_state=191
    )

rmse_scores, mae_scores = T.rolling_window_cv(
    train_days=365 * 2,
    gap_days=21,
    test_days=60,
    df=three_exo_df,
    model=model
)

mean_rmse = np.mean(rmse_scores)
se_rmse = np.std(rmse_scores)

mean_mae = np.mean(mae_scores)
se_mae = np.std(mae_scores)

print("Summary of Performance Statistics")
print(f"RMSE: {mean_rmse} +/- {se_rmse}")
print(f"MAE: {mean_mae} +/- {se_mae}")

Fold 1
2018-07-05 00:00:00 2020-07-19 00:00:00
2020-08-10 00:00:00 2020-10-08 00:00:00
RMSE: 1.5536426248101705
MAE: 0.9872114042222759
Fold 2
2018-09-03 00:00:00 2020-09-17 00:00:00
2020-10-09 00:00:00 2020-12-07 00:00:00
RMSE: 5.130774402256042
MAE: 2.3249963269864984
Fold 3
2018-11-02 00:00:00 2020-11-16 00:00:00
2020-12-08 00:00:00 2021-02-05 00:00:00
RMSE: 7.061683453263363
MAE: 5.158533375982518
Fold 4
2019-01-01 00:00:00 2021-01-15 00:00:00
2021-02-06 00:00:00 2021-04-18 00:00:00
RMSE: 12.791956261369904
MAE: 9.155188886462732
Fold 5
2019-03-04 00:00:00 2021-03-28 00:00:00
2021-04-19 00:00:00 2021-06-17 00:00:00
RMSE: 4.04122790914381
MAE: 2.352153024447235
Fold 6
2019-05-03 00:00:00 2021-05-27 00:00:00
2021-06-18 00:00:00 2021-08-16 00:00:00
RMSE: 0.9971107695380308
MAE: 0.771534574960101
Fold 7
2019-07-02 00:00:00 2021-07-26 00:00:00
2021-08-17 00:00:00 2021-10-15 00:00:00
RMSE: 1.3227786772220909
MAE: 0.9607922564232872
Fold 8
2019-08-31 00:00:00 2021-09-24 00:00:00
2021-10-1

## 12 Neighbors + Exo

In [116]:
twelve_neighbors_exo_df = exp_1_df[['row', 'col', 'date', 'pm25_t', 'u_wind_10m', 'v_wind_10m',
                                    'dew_temp_2m', 'temp_2m', 'surf_pressure', 'precip_sum', 'frp',
                                    'elevation', 'delta_pm25_t',  
                                    'delta_pm25_t_r1_c0', 'delta_pm25_t_r-1_c0',
                                    'delta_pm25_t_r0_c1', 'delta_pm25_t_r0_c-1',
                                    'delta_pm25_t_r1_c1', 'delta_pm25_t_r1_c-1',
                                    'delta_pm25_t_r-1_c1', 'delta_pm25_t_r-1_c-1',
                                    'delta_pm25_t_r-2_c0', 'delta_pm25_t_r2_c0',
                                    'delta_pm25_t_r0_c-2', 'delta_pm25_t_r0_c2',
                                    'delta_pm25_t+1']]
twelve_neighbors_exo_df.head()

,row,col,date,pm25_t,u_wind_10m,v_wind_10m,dew_temp_2m,temp_2m,surf_pressure,precip_sum,...,delta_pm25_t_r0_c-1,delta_pm25_t_r1_c1,delta_pm25_t_r1_c-1,delta_pm25_t_r-1_c1,delta_pm25_t_r-1_c-1,delta_pm25_t_r-2_c0,delta_pm25_t_r2_c0,delta_pm25_t_r0_c-2,delta_pm25_t_r0_c2,delta_pm25_t+1
112320,2,2,2018-07-05,8.506299,1.099417,0.845259,292.53455,297.36017,88500.48,0.001086,...,1.526387,0.119964,-0.102599,0.420908,1.152598,0.973524,0.745076,0.531390,-0.076599,-1.214631
112321,2,2,2018-07-06,7.291668,0.536112,0.847002,292.94757,296.90536,88492.29,0.002184,...,-2.001883,0.452158,-0.351426,-0.559418,-1.761432,-1.125940,-0.788587,-0.279049,0.366528,1.127308
112322,2,2,2018-07-07,8.418976,0.653129,0.892963,293.46637,296.09024,88523.93,0.010981,...,1.450109,0.063593,1.017430,0.708406,1.741480,0.587489,0.626072,0.490491,-0.134154,0.610699
112323,2,2,2018-07-08,9.029675,0.758944,0.372737,293.11792,295.39346,88557.59,0.007404,...,0.941031,0.888477,0.539345,0.311186,0.926645,0.058055,1.012921,1.268657,0.875981,0.086799
112324,2,2,2018-07-09,9.116474,0.735163,0.694967,292.74075,295.87690,88641.15,0.001646,...,-0.670372,-0.533840,-0.117224,-0.060415,-0.514772,0.625493,-0.704665,-1.068175,-0.195938,0.337736


In [117]:
model = xgb.XGBRegressor(
        objective='reg:squarederror',
        **params,
        random_state=191
    )

rmse_scores, mae_scores = T.rolling_window_cv(
    train_days=365 * 2,
    gap_days=21,
    test_days=60,
    df=twelve_neighbors_exo_df,
    model=model
)

mean_rmse = np.mean(rmse_scores)
se_rmse = np.std(rmse_scores)

mean_mae = np.mean(mae_scores)
se_mae = np.std(mae_scores)

print("Summary of Performance Statistics")
print(f"RMSE: {mean_rmse} +/- {se_rmse}")
print(f"MAE: {mean_mae} +/- {se_mae}")

Fold 1
2018-07-05 00:00:00 2020-07-19 00:00:00
2020-08-10 00:00:00 2020-10-08 00:00:00
RMSE: 1.5404586950514998
MAE: 0.9822771582189067
Fold 2
2018-09-03 00:00:00 2020-09-17 00:00:00
2020-10-09 00:00:00 2020-12-07 00:00:00
RMSE: 5.249185006264551
MAE: 2.3656162938731784
Fold 3
2018-11-02 00:00:00 2020-11-16 00:00:00
2020-12-08 00:00:00 2021-02-05 00:00:00
RMSE: 7.054573179260198
MAE: 5.17452007426294
Fold 4
2019-01-01 00:00:00 2021-01-15 00:00:00
2021-02-06 00:00:00 2021-04-18 00:00:00
RMSE: 12.830661413282938
MAE: 9.15042550298882
Fold 5
2019-03-04 00:00:00 2021-03-28 00:00:00
2021-04-19 00:00:00 2021-06-17 00:00:00
RMSE: 4.041923543753887
MAE: 2.348736256764067
Fold 6
2019-05-03 00:00:00 2021-05-27 00:00:00
2021-06-18 00:00:00 2021-08-16 00:00:00
RMSE: 0.9946205788535153
MAE: 0.7704724077944024
Fold 7
2019-07-02 00:00:00 2021-07-26 00:00:00
2021-08-17 00:00:00 2021-10-15 00:00:00
RMSE: 1.3482607908673883
MAE: 0.9630094326341432
Fold 8
2019-08-31 00:00:00 2021-09-24 00:00:00
2021-10-1

## 5x5 + Exo

In [119]:
five_exo_df = exp_1_df[['row', 'col', 'date', 'pm25_t', 'u_wind_10m', 'v_wind_10m',
                        'dew_temp_2m', 'temp_2m', 'surf_pressure', 'precip_sum', 'frp',
                        'elevation', 'delta_pm25_t',  
                        'delta_pm25_t_r-2_c-2', 'delta_pm25_t_r-2_c-1', 'delta_pm25_t_r-2_c0', 'delta_pm25_t_r-2_c1', 'delta_pm25_t_r-2_c2',
                        'delta_pm25_t_r-1_c-2', 'delta_pm25_t_r-1_c-1', 'delta_pm25_t_r-1_c0', 'delta_pm25_t_r-1_c1', 'delta_pm25_t_r-1_c2',
                        'delta_pm25_t_r0_c-2', 'delta_pm25_t_r0_c-1', 'delta_pm25_t_r0_c1', 'delta_pm25_t_r0_c2',
                        'delta_pm25_t_r1_c-2', 'delta_pm25_t_r1_c-1', 'delta_pm25_t_r1_c0', 'delta_pm25_t_r1_c1', 'delta_pm25_t_r1_c2',
                        'delta_pm25_t_r2_c-2', 'delta_pm25_t_r2_c-1', 'delta_pm25_t_r2_c0', 'delta_pm25_t_r2_c1', 'delta_pm25_t_r2_c2',
                        'delta_pm25_t+1'
]]

five_exo_df.head()

,row,col,date,pm25_t,u_wind_10m,v_wind_10m,dew_temp_2m,temp_2m,surf_pressure,precip_sum,...,delta_pm25_t_r1_c-1,delta_pm25_t_r1_c0,delta_pm25_t_r1_c1,delta_pm25_t_r1_c2,delta_pm25_t_r2_c-2,delta_pm25_t_r2_c-1,delta_pm25_t_r2_c0,delta_pm25_t_r2_c1,delta_pm25_t_r2_c2,delta_pm25_t+1
112320,2,2,2018-07-05,8.506299,1.099417,0.845259,292.53455,297.36017,88500.48,0.001086,...,-0.102599,0.198054,0.119964,0.473723,0.354453,0.733201,0.745076,0.145992,0.144983,-1.214631
112321,2,2,2018-07-06,7.291668,0.536112,0.847002,292.94757,296.90536,88492.29,0.002184,...,-0.351426,-0.564943,0.452158,0.313466,-0.011651,-0.960510,-0.788587,0.048550,0.975503,1.127308
112322,2,2,2018-07-07,8.418976,0.653129,0.892963,293.46637,296.09024,88523.93,0.010981,...,1.017430,1.057073,0.063593,-0.459555,0.387937,1.195193,0.626072,0.028603,-0.755823,0.610699
112323,2,2,2018-07-08,9.029675,0.758944,0.372737,293.11792,295.39346,88557.59,0.007404,...,0.539345,0.560488,0.888477,1.368596,0.601076,0.385827,1.012921,0.865992,1.554551,0.086799
112324,2,2,2018-07-09,9.116474,0.735163,0.694967,292.74075,295.87690,88641.15,0.001646,...,-0.117224,-0.156699,-0.533840,-0.700388,-0.497408,-0.014912,-0.704665,-0.464011,-1.017161,0.337736


In [120]:
model = xgb.XGBRegressor(
        objective='reg:squarederror',
        **params,
        random_state=191
    )

rmse_scores, mae_scores = T.rolling_window_cv(
    train_days=365 * 2,
    gap_days=21,
    test_days=60,
    df=five_exo_df,
    model=model
)

mean_rmse = np.mean(rmse_scores)
se_rmse = np.std(rmse_scores)

mean_mae = np.mean(mae_scores)
se_mae = np.std(mae_scores)

print("Summary of Performance Statistics")
print(f"RMSE: {mean_rmse} +/- {se_rmse}")
print(f"MAE: {mean_mae} +/- {se_mae}")

Fold 1
2018-07-05 00:00:00 2020-07-19 00:00:00
2020-08-10 00:00:00 2020-10-08 00:00:00
RMSE: 1.5355051141476084
MAE: 0.9778407375318633
Fold 2
2018-09-03 00:00:00 2020-09-17 00:00:00
2020-10-09 00:00:00 2020-12-07 00:00:00
RMSE: 5.277962572744154
MAE: 2.35861406371097
Fold 3
2018-11-02 00:00:00 2020-11-16 00:00:00
2020-12-08 00:00:00 2021-02-05 00:00:00
RMSE: 7.072864608278142
MAE: 5.173320041267818
Fold 4
2019-01-01 00:00:00 2021-01-15 00:00:00
2021-02-06 00:00:00 2021-04-18 00:00:00
RMSE: 12.793278681992668
MAE: 9.110088867753948
Fold 5
2019-03-04 00:00:00 2021-03-28 00:00:00
2021-04-19 00:00:00 2021-06-17 00:00:00
RMSE: 4.015516507489309
MAE: 2.3350350208749475
Fold 6
2019-05-03 00:00:00 2021-05-27 00:00:00
2021-06-18 00:00:00 2021-08-16 00:00:00
RMSE: 0.9848992041582701
MAE: 0.763735042721831
Fold 7
2019-07-02 00:00:00 2021-07-26 00:00:00
2021-08-17 00:00:00 2021-10-15 00:00:00
RMSE: 1.2872082807385457
MAE: 0.9477887194616236
Fold 8
2019-08-31 00:00:00 2021-09-24 00:00:00
2021-10-1